In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/FIRST-J110949.3+124614_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/FIRST-J110949.3+124614_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     336   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        22   (2094, 964)   float32   


In [3]:
image_cut = image[49:149, 0:1890]
image_error_cut = image_error[49:149, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.04263
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5, params6):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) +
        sersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/PSF/Real_seeing_FIRST_1.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, params6, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5, params6)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, params6, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, params6, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

36.439774258506624
44.8688474504367
42.024184163860774
63.550701956414315
99.02578521759608
97.94444345895607
41.30077442707752
41.18457411383642
45.05452261327962
49.15972543448083
6.295707980528358
6.329376093375969
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.981513e+01  3.161259e-01   0.028620   0.041920   0.017846   
Green      1.436196e-01  9.814003e+01   0.131949   0.222103   0.059558   
Magenta    4.070366e-02  1.526537e+00  67.254132  58.251614  20.347614   
Orange     2.390039e-28  4.123876e-04  29.959586  40.047980   4.100354   
Chocolate  5.470053e-04  1.689759e-02   2.625702   1.436381  75.472656   
Purple     6.643461e-24  9.339736e-14   0.000012   0.000002   0.001971   

                 Peak 6  
Blue       3.647219e-02  
Green      7.601487e-02  
Magenta    6.319102e+00  
Orange     1.157853e-08  
Chocolate  1.128881e+00  
Purple     9.243953e+01  


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

101.18196827362637
122.54992134877983
114.81722751027988
169.54812286249256
188.86190233619374
187.35862347818727
124.01414310747991
123.56608487704831
127.56564261835013
137.9266982496337
18.273310995177194
18.357273237535814
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.979928e+01  3.485799e-01   0.030121   0.041963   0.018937   
Green      1.559153e-01  9.794501e+01   0.139831   0.223268   0.063326   
Magenta    4.421521e-02  1.687075e+00  65.982377  58.565528  21.749566   
Orange     2.026978e-27  6.416030e-04  30.898358  39.697348   4.534603   
Chocolate  5.935021e-04  1.869656e-02   2.949298   1.471891  73.631260   
Purple     9.210708e-24  1.242286e-13   0.000015   0.000003   0.002307   

                 Peak 6  
Blue       3.773632e-02  
Green      7.870689e-02  
Magenta    6.571746e+00  
Orange     2.372498e-08  
Chocolate  1.181499e+00  
Purple     9.213031e+01  


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3)
    .reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

140.71984376559314
164.74759270038913
154.86763737886645
217.08389612957438
267.490635545001
265.8747341985554
206.05241026093208
205.0256948599819
188.11815539281193
199.60413450988673
28.08518693388346
28.1774467258565
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.975631e+01  4.356391e-01   0.032093   0.042310   0.021489   
Green      1.892227e-01  9.741886e+01   0.150718   0.227706   0.072246   
Magenta    5.374430e-02  2.120390e+00  64.433775  59.181822  25.170137   
Orange     3.573564e-26  1.541368e-03  31.867952  38.971283   5.692441   
Chocolate  7.192167e-04  2.357076e-02   3.515440   1.576876  69.040392   
Purple     1.871510e-23  2.339162e-13   0.000023   0.000003   0.003294   

                 Peak 6  
Blue       4.103472e-02  
Green      8.575439e-02  
Magenta    7.245148e+00  
Orange     9.094797e-08  
Chocolate  1.325231e+00  
Purple     9.130283e+01  


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

13.724049741723025
11.123655400395466
156.8539045718274
68.75633993012676
36.91154196935887
36.79235947308696
14.082386512486492
14.069166161545343
11.38080136256126
11.402719524902125
1.7734994831590316
1.9170072293633122
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.792346e+01  4.139781e-02   0.042743   0.067832   0.040239   
Green      9.937071e-02  9.939294e+01   0.079440   0.157059   0.043566   
Magenta    1.977133e+00  5.606038e-01  66.511557  46.739143  33.454798   
Orange     5.138907e-13  5.021156e-03  33.082213  52.978954   8.271330   
Chocolate  1.448039e-15  1.643724e-09   0.278273   0.051813  58.156982   
Purple     3.687332e-05  3.478481e-05   0.005773   0.005200   0.033084   

              Peak 6  
Blue        0.076627  
Green       0.033634  
Magenta    45.976642  
Orange      0.000198  
Chocolate   0.000372  
Purple     53.912527  


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

38.20547530778019
31.317568836873168
420.36452542018384
186.69937269095485
70.65563596614487
70.2803007631916
42.5576846595595
42.527080451981476
32.71781310073727
32.73203408946922
5.168967873046015
5.579580654484291
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.776037e+01  4.650838e-02   0.044806   0.067454   0.042049   
Green      1.078766e-01  9.931891e+01   0.084245   0.157262   0.045692   
Magenta    2.131712e+00  6.283304e-01  65.245368  47.605675  35.177501   
Orange     8.697243e-13  6.210540e-03  34.232102  52.104773   8.856451   
Chocolate  1.731336e-15  2.067294e-09   0.387365   0.059642  55.843426   
Purple     3.981584e-05  3.906306e-05   0.006114   0.005193   0.034882   

              Peak 6  
Blue        0.078950  
Green       0.034710  
Magenta    47.378916  
Orange      0.000248  
Chocolate   0.000432  
Purple     52.506745  


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


52.369357213469364
44.28260064292215
512.5947618251163
237.24098200434486
99.42116480810039
98.91678894846333
72.07627997469258
72.02334147582832
49.424277876833536
49.390392552518776
7.967504036789309
8.510801713449647
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.727088e+01  6.418630e-02   0.048027   0.066692   0.046564   
Green      1.337240e-01  9.906343e+01   0.092064   0.158407   0.051098   
Magenta    2.595351e+00  8.616373e-01  63.321226  49.664831  39.651383   
Orange     2.464044e-12  1.069654e-02  35.895880  50.022419  10.433262   
Chocolate  2.691257e-15  3.711174e-09   0.636138   0.082457  49.778156   
Purple     4.866808e-05  5.384269e-05   0.006665   0.005193   0.039536   

              Peak 6  
Blue        0.085590  
Green       0.037793  
Magenta    51.387576  
Orange      0.000423  
Chocolate   0.000629  
Purple     48.487988  


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

7.507363376766871
8.321273484916984
34.71943818224953
59.16510857295934
57.76796971086543
53.9840978716872
19.680620329207326
19.621741680353306
18.45060154521598
19.109793952306823
1.9881764767507533
1.9968609566539897
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.520080e+01   0.158839   0.021221   0.038064   0.018757   0.048219
Green      1.958491e-01  96.688880   0.064832   0.140130   0.036852   0.048458
Magenta    4.603327e+00   3.146766  60.171457  41.287533  26.217151  51.738142
Orange     1.481322e-18   0.005417  38.921188  58.199341  17.941230   0.000203
Chocolate  7.526167e-09   0.000020   0.818554   0.332273  55.769015   0.054407
Purple     2.076325e-05   0.000077   0.002749   0.002659   0.016995  48.110572


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

21.16486888078189
23.22176871748581
94.31227289113022
156.51656187707292
110.72215043647408
104.01337824831494
59.68984446163824
59.33123755486014
53.36837910909029
55.028646192202714
5.760658620465302
5.775038227150613
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.488944e+01   0.176092   0.022223   0.037723   0.019484   0.049980
Green      2.096902e-01  96.337129   0.068562   0.139697   0.038391   0.050284
Magenta    4.900847e+00   3.479539  58.831204  42.262617  27.412632  53.630354
Orange     4.356799e-18   0.007132  40.092500  57.206534  18.845973   0.000281
Chocolate  8.276234e-09   0.000023   0.982596   0.350780  53.665692   0.059075
Purple     2.215838e-05   0.000085   0.002914   0.002650   0.017828  46.210027


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

30.621567003650853
32.91681053328023
125.20943872114864
195.94163227674085
155.25530270958927
147.44944163881016
102.206977175061
100.68461530561315
82.32087409305575
84.1628897436115
8.847667021788638
8.853926543037396
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.410179e+01   0.223349   0.023915   0.036904   0.021138   0.054394
Green      2.454853e-01  95.381049   0.075003   0.138883   0.041978   0.054892
Magenta    5.652699e+00   4.382386  56.595250  44.674111  30.307787  58.376230
Orange     2.415509e-17   0.013075  41.985941  54.746400  21.020060   0.000576
Chocolate  1.036203e-08   0.000032   1.316690   0.401073  48.589193   0.072672
Purple     2.572029e-05   0.000108   0.003201   0.002630   0.019844  41.441236


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/Fit/SII_fit.csv", index_col = 0)

In [21]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6
I_e,0.4602,2.9310,0.3676,4.8578,1.7656,0.1000
r_e,10.0000,3.4375,49.9989,6.6840,2.1825,5.0000
n,1.8839,3.4926,2.5119,0.3622,0.9496,1.7431
x_0,16.5619,41.5775,65.9756,63.7285,72.9746,91.3193


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.2370352824111235
6.683063455972647
36.29322999602115
64.48615210922978
49.521358182017735
46.069589619375286
15.84535612314713
15.790167857815746
13.143547116603159
12.922509227340935
1.924093266688413
1.9877327928875177
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.369075e+01  1.497619e-01   0.020602   0.040134   0.020623   
Green      2.907352e-01  9.660153e+01   0.092195   0.208302   0.066427   
Magenta    6.018504e+00  3.248192e+00  69.968072  53.104396  38.734294   
Orange     8.743378e-34  4.723844e-04  29.800056  46.621466  23.172107   
Chocolate  2.017085e-19  1.640162e-10   0.116371   0.022984  37.985029   
Purple     1.267245e-05  4.873951e-05   0.002703   0.002717   0.021520   

              Peak 6  
Blue        0.033312  
Green       0.072028  
Magenta    57.719260  
Orange      0.000002  
Chocolate   0.000223  
Purple     42.175176  


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

17.731059264944083
18.855946912310564
98.54115362901213
170.30497791708834
94.67177706515264
88.55292762045954
48.273255746209124
47.942500665887245
38.468579948251204
37.9041597994959
5.610607468272893
5.769608764230973
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.333730e+01  1.661540e-01   0.021645   0.039609   0.021177   
Green      3.085304e-01  9.623976e+01   0.097643   0.206605   0.068362   
Magenta    6.354156e+00  3.593276e+00  68.749754  53.964305  40.010331   
Orange     2.642777e-32  7.570703e-04  30.964925  45.759970  23.880319   
Chocolate  2.615013e-19  2.198154e-10   0.163154   0.026813  35.997483   
Purple     1.341930e-05  5.412172e-05   0.002878   0.002698   0.022328   

              Peak 6  
Blue        0.034319  
Green       0.074250  
Magenta    59.448860  
Orange      0.000003  
Chocolate   0.000272  
Purple     40.442296  


In [24]:
_HAdata1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

26.126758684965854
27.37581445824525
130.72623105753652
212.4171247312131
132.17485545969356
125.05858658983215
83.54862527147317
82.15736711532173
61.15808360337656
60.48294168808628
8.76747017682347
8.942793942572717
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.333730e+01  2.110599e-01   0.023419   0.038370   0.022309   
Green      3.085304e-01  9.525829e+01   0.107084   0.202874   0.072446   
Magenta    6.354156e+00  4.528685e+00  66.723851  56.057397  42.870642   
Orange     2.642777e-32  1.898903e-03  32.873988  43.661007  25.357756   
Chocolate  2.615013e-19  4.224353e-10   0.268474   0.037697  31.652686   
Purple     1.341930e-05  6.889120e-05   0.003183   0.002654   0.024162   

              Peak 6  
Blue        0.036737  
Green       0.079613  
Magenta    63.596433  
Orange      0.000011  
Chocolate   0.000435  
Purple     36.286771  


## OIII

In [25]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Paola_red/Fit/OIII_fit.csv", index_col = 0)

In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

19.077030905257754
18.184323461345777
91.66145412896942
70.69836262597858
36.630993139797965
34.332337025436786
11.932740213352949
11.893480054955361
7.749844150543333
7.7937918896231695
2.235391818712051
2.4906470796481246
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.801478e+01  8.752595e-02   0.061411   0.112277   0.088580   
Green      1.108421e-01  9.870199e+01   0.127432   0.285435   0.111698   
Magenta    1.874335e+00  1.210378e+00  76.622060  61.318842  58.910996   
Orange     4.114718e-76  2.637071e-05  23.147409  38.274056   6.860800   
Chocolate  1.260169e-36  1.077486e-17   0.035437   0.002627  33.979847   
Purple     4.295567e-05  7.929731e-05   0.006251   0.006764   0.048079   

                 Peak 6  
Blue       1.035608e-01  
Green      5.671414e-02  
Magenta    4.680504e+01  
Orange     1.515270e-18  
Chocolate  1.233959e-07  
Purple     5.303469e+01  


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

52.87838104040687
50.333469198515516
247.14958099636013
190.4924842551207
69.8805604427344
65.80668121608281
36.353995530223536
36.12713378342294
22.840628065360715
22.95240036092483
6.426273722425531
7.07155606740859
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.784968e+01  9.771277e-02   0.064564   0.110727   0.090270   
Green      1.207142e-01  9.855390e+01   0.135317   0.283171   0.114183   
Magenta    2.029564e+00  1.348237e+00  75.798900  62.003530  60.385631   
Orange     1.029971e-70  6.242870e-05  23.932321  37.592150   7.399787   
Chocolate  3.211214e-36  2.302684e-17   0.062266   0.003726  31.960741   
Purple     4.656988e-05  8.847062e-05   0.006632   0.006696   0.049388   

                 Peak 6  
Blue       1.081571e-01  
Green      5.931194e-02  
Magenta    4.889602e+01  
Orange     4.250345e-17  
Chocolate  2.031861e-07  
Purple     5.093652e+01  


In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

71.89670340861701
68.66219413723735
308.2285602210708
239.9398267071837
97.27624696707582
92.54309049516145
62.912837241436804
61.978584578417184
36.861395278798874
36.98876010420626
9.719127036236397
10.440838113969264
                 Peak 1        Peak 2     Peak 3     Peak 4     Peak 5  \
Blue       9.735863e+01  1.317768e-01   0.069900   0.107070   0.093515   
Green      1.504114e-01  9.806058e+01   0.148961   0.278263   0.119279   
Magenta    2.490900e+00  1.807259e+00  74.420594  63.651390  63.603111   
Orange     2.846461e-65  2.648175e-04  25.221586  35.949455   8.745492   
Chocolate  1.613419e-35  9.632969e-17   0.131671   0.007281  27.386364   
Purple     5.734006e-05  1.190930e-04   0.007288   0.006541   0.052239   

                 Peak 6  
Blue       1.194418e-01  
Green      6.574051e-02  
Magenta    5.403868e+01  
Orange     2.261769e-15  
Chocolate  5.764201e-07  
Purple     4.577614e+01  
